### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="wine_world_cost",
    dataset_year="2023",
    domain_str="business & marketing",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/datasets/elvinrustam/wine-dataset",
    download_description="""
There are many different wine datasets available, but this one is a comprehensive collection of wine-related data that includes information on various wine characteristics and text. So we stick to it.

kaggle datasets download elvinrustam/wine-dataset && unzip wine-dataset.zip && rm wine-dataset.zip
mkdir -p local-data-warehouse/wine_world_cost && mv WineDataset.csv local-data-warehouse/wine_world_cost
""",
    # References
    academic_reference_bibtex=r"""@misc{Rustamov2023WineDataset,
  author = {Elvin Rustamov},
  title  = {Wine Dataset},
  year   = {2023},
  howpublished = {\url{https://www.kaggle.com/datasets/elvinrustam/wine-dataset}},
  note   = {Kaggle dataset}
}
""",
    academic_reference_bibtex_key="Rustamov2023WineDataset",
    license="CC0: Public Domain",
    data_tags=["IID"],
    curation_comments="""
We start with the Kaggle version. We resolve several issues from the web scrapper.

- We create a task to predict the price per bottle. We remove the 11 rows that have price information per case or each.
- Create the price target by parsing the price column.
- We map capacity to numeric values in ML.
- We parse the ABV to a numeric value
- We parse the vintage to a numeric date value, ensuring that each wine maps to the oldest year in its special syntax.
- We log scale the target, as the price distribution is very skewed.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="Price",
    problem_type="regression",
    objective_metric_name="rmse",
)

## Preprocessing

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv(dataset_mold.path / "WineDataset.csv")
print("Loaded data shape:", df.shape)

df = df[df["Per bottle / case / each"] == "per bottle"].reset_index(drop=True)

df["Price"] = df["Price"].apply(lambda x: x.replace("£", "").replace(" per bottle", "")).astype(float)
df["ABV%"] = df["ABV"].apply(lambda x: str(x).replace("ABV ", "").replace("%", "")).astype(float)

df["Capacity"] = df["Capacity"].map({
    "75CL": 750,
    "37.5CL": 375,
    "1.5LTR": 1500,
    "750ML": 750,
    "150CL": 1500,
    "50CL": 500,
    "70CL": 700,
    "500ML": 500,
    "375ML": 375,
    "300CL": 3000,
}).astype(float)
df["VintageYear"] = pd.to_datetime(df["Vintage"].apply(lambda x: x.split("/")[0]).replace("NV", np.nan)).dt.year

df = df.drop(columns=[
    # Constant
    "Per bottle / case / each",
    # Meaning of feature is missing, and it is unclear how to interpret it
    "Unit",
    # Copied columns
    "ABV",
    "Vintage",
])

as_cat_type = [
    "Closure",
    "Type",
    "Style",
]
as_string_type = [
    "Title",
    "Description",
    "Grape",
    "Secondary Grape Varieties",
    "Country",
    "Characteristics",
    "Region",
    "Appellation",
]

for c in as_string_type:
    nan_mask = df[c].isna()
    df.loc[nan_mask, c] = np.nan
    df[c] = df[c].astype("string")

df[as_cat_type] = df[as_cat_type].astype("category")

# Log scale target
df["Price"] = np.log(df["Price"])
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

Loaded data shape: (1290, 17)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 1,279
Columns: 15

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,Title,Description,Price,Capacity,Grape,Secondary Grape Varieties,Closure,Country,Characteristics,Type,Region,Style,Appellation,ABV%,VintageYear
0,"Taylor's Port 2007, Portugal",<NA>,4.442651,750.0,Tinta Barroca,"Touriga Nacional, Tinta Amarela, Tinta Roriz, Touriga Franca",Natural Cork,Portugal,"Jammy, Chocolate, Dried Fruit",Red,Douro,NaN,<NA>,20.0,2007.0
1,"Philippe le Hardi 'Les Platanes d'Henri IV' 2019/20, Santenay","This is made on a magnificent 9th century estate that's named after the first Duke of Burgundy, Philippe le Hardi. He, interestingly, became famous when he banned Gamay from the region in 1395. Today, this is the hub of environmentally friendly, excellently produced wines, like this Santenay. It's rich, ripe and complex, with notes of stone fruits, vanilla and honey. Serve lightly chilled with Camembert.",3.610648,750.0,Chardonnay,<NA>,Natural Cork,France,"Vanilla, Bread, Cream, Stone Fruit",White,Burgundy,Rich & Toasty,Santenay,13.5,2020.0
2,"Cave Vinicole de Hunawihr Kuhlmann-Platz 'Cuvée Prestige' Pinot Noir 2021/22, Alsace","Cave Vinicole de Hunawihr began in Alsace, on the German border of France, in 1954. Their aim was to restore the region’s winemaking industry after it was badly damaged by the German occupation in World War Two. Today, this cooperative consists of 110 members, and it's safe to say they've smashed their initial target. This is a light and fresh Pinot Noir, with flavours of cherry, strawberry, sweet spices and cedar. Try it with turkey or chicken.",2.638343,750.0,Pinot Noir,<NA>,Screwcap,France,"Red Fruit, Black Cherry, Blackcurrant, Cranberry, Red Cherry",Red,Alsace,Light & Elegant,<NA>,13.5,2022.0
3,"Copper Kingdom Shiraz 2017, Barossa","Barossa is where you’ll find some of Australia’s most sought-after wines. The region’s plentiful sunshine means the Shiraz here is big, bold and packed with intense black fruit flavour. Barossa was once famed for its metal-rich soils, but now it's flavourful red wines like Copper Kingdom that dominate the headlines. Copper Kingdom is a rich Shiraz with notes of black pepper and ripe dark fruits. Partner with a sizzling barbecued steak.",2.707383,750.0,Shiraz,<NA>,Screwcap,Australia,"Leather, Black Pepper, Blackberry, Blackcurrant, Blueberry, Chocolate, Jammy",Red,South Australia,Bold & Spicy,Barossa Valley,14.5,2017.0
4,"The King's Wrath Pinot Noir 2020/21, Marlborough","Brent Marris is the man behind our bestselling white wine, The Ned. And he's exceptional at making reds too. Like Sauvignon Blanc, another thin-skinned grape, Pinot Noir thrives in Marlborough's cool climate and long ripening season. As a result, this is a rich and full of fruit. Expect blackberry, raspberry, plum and toasted oak. Try it with rich fish, like tuna or trout.",2.771964,750.0,Pinot Noir,<NA>,Screwcap,New Zealand,"Sweet Spice, Black Cherry, Blackberry, Red Fruit",Red,Marlborough,Light & Elegant,<NA>,14.0,2021.0


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,Style,category,71,5.55,16,"Savoury & Full Bodied, Rich & Toasty, Bold & Spicy, Light & Elegant, Crisp & Zesty, Fresh & Elegant , Ripe & Rounded, Crisp & Fruity, Rich & Juicy, Smooth & Mellow"
1,Type,category,4,0.31,6,"White, Red, Rosé, Tawny, Orange, Brown"
2,Closure,category,0,0.00,4,"Natural Cork, Screwcap, Synthetic Cork, Vinolok"
3,VintageYear,float64,170,13.29,21,"2022.0, 2021.0, 2020.0, 2019.0, 2018.0, 2017.0, 2016.0, 2015.0, 2014.0, 2023.0"
4,ABV%,float64,3,0.23,32,"13.5, 13.0, 14.0, 14.5, 12.5, 12.0, 15.0, 11.0, 11.5, 20.0"
5,Price,float64,0,0.00,119,"2.4841, 2.7074, 2.3016, 2.6383, 2.8326, 2.772, 2.5642, 2.9952, 2.8898, 3.4009"
6,Capacity,float64,0,0.00,6,"750.0, 1500.0, 375.0, 500.0, 700.0, 3000.0"
7,Secondary Grape Varieties,string,793,62.00,197,"Pinot Noir, Pinot Meunier, Pinot Noir, Merlot, Syrah, Chardonnay, Cabernet Sauvignon, Cabernet Franc, Mourvèdre, Syrah, Pinot Meunier, Chardonnay, Sémillon"
8,Appellation,string,635,49.65,179,"Rioja, Barossa Valley, Chablis, Sancerre, Côtes Du Rhône, Côtes De Provence, Uco Valley, Puligny-Montrachet, Napa Valley, Pays D'Oc"
9,Region,string,159,12.43,94,"Burgundy, Bordeaux, Marlborough, Loire, South Australia, Rhône, Languedoc-Roussillon, California, Rioja And Navarra, Mendoza"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
Price,1279.0,3.027286,0.694994,1.607436,6.063785
Capacity,1279.0,759.460516,140.785100,375.000000,3000.000000
ABV%,1276.0,13.429702,1.814146,0.500000,40.000000
VintageYear,1109.0,2019.853021,2.806491,1999.000000,2023.000000


In [7]:
# Categorical Feature Statistics
cat_stats

value  \
column                    rank                                                                        
Appellation               1                                                                    <NA>   
                          2                                                                   Rioja   
                          3                                                          Barossa Valley   
                          4                                                                 Chablis   
                          5                                                                Sancerre   
Characteristics           1                                                                    <NA>   
                          2                                            Strawberry, Peach, Raspberry   
                          3                                        Green Apple, Citrus Fruit, Grass   
                          4                                         Vanilla, Black Fruit, Red Fruit   
                          5                        Sweet Spice, Black Cherry, Blackberry, Red Fruit   
Closure                   1                                                            Natural Cork   
                          2                                                                Screwcap   
                          3                                                          Synthetic Cork   
                          4                                                                 Vinolok   
Country                   1                                                                  France   
                          2                                                                   Italy   
                          3                                                                   Spain   
                          4                                                               Australia   
                          5                                                             New Zealand   
Description               1                                                                    <NA>   
                          2     Nicolás Catena was the first South American winemaker to ever be...   
                          3     Grand Cru Chardonnay has been used to make this Champagne. Citru...   
                          4     The Definition range brings the world's greatest wine styles to ...   
                          5     The Reverdy-Ducroux family has been cultivating its vines on the...   
Grape                     1                                                              Chardonnay   
                          2                                                              Pinot Noir   
                          3                                                         Sauvignon Blanc   
                          4                                                      Cabernet Sauvignon   
                          5                                                                Grenache   
Region                    1                                                                    <NA>   
                          2                                                                Burgundy   
                          3                                                                Bordeaux   
                          4                                                             Marlborough   
                          5                                                                   Loire   
Secondary Grape Varieties 1                                                                    <NA>   
                          2                                               Pinot Noir, Pinot Meunier   
                          3                                                              Pinot Noir   
                          4                                                  

In [8]:
# Target Distribution
target_df

,y_missing_count,non_positive_pct,skew_y,skew_log,var_y,var_log,log_used,aic_exponential,aic_lognormal,dist_hint
0,0,0.0,1.138,0.518,0.483,0.046,log,6352.7,5922.4,lognormal


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_iid_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_iid_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=10, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
019c32f6-9391-7812-b543-66fbb299dc51
0fe56195a830da8b4e3b55d75c79dd1c5495dff159e969d0503dbfb5772c0a04
